# ODA-LAB sunset-member-index

**Runtime → Run all.** Then allow Drive when the popup hits.

This is not a prompt dump. Cell 2 mounts. Cell 3 walks MyDrive for the
four small sunset packs and prints a member list when it can open them.
It will not touch the 416.6 MB grok-five tar.

Shelf: `1ypcn8_RExguvTGJVLgaVu37JPFr9fjlj` (`12_ODA-LAB-NOTEBOOKS`)


In [ ]:
# Cell 1 — mount
from google.colab import drive
from pathlib import Path
import os, subprocess, json, time

print('=== ODA LAB mount ===')
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive')
print('mounted', ROOT.exists(), ROOT)


In [ ]:
# Cell 2 — find the four small packs and list members
PACKS = [
  {
    "n": 18,
    "name": "sunset_full_2026-09-10.zip",
    "id": "14vU0nA6IwQGbzXXb7G6ZALBsnYkD0Fhg",
    "bytes": 35981
  },
  {
    "n": 19,
    "name": "2026-09-10_SUNSET_TEXT_PACKAGES.zip",
    "id": "1h68ZxmzfpGlxykGE1orYmZZyFm2Vxlms",
    "bytes": 73695
  },
  {
    "n": 5,
    "name": "sunset_pixels_other_2026-09-10.zip",
    "id": "1o2Ax6cuFTwu9d9XujuoUlBGtozH5s0ef",
    "bytes": 7801401
  },
  {
    "n": 17,
    "name": "sunset_full_archive_2026-09-10.zip",
    "id": "17Fhlp5gm0EzPrI7j6bnzHTzzGtVH5ZG0",
    "bytes": 6300000
  }
]
JOB = 'sunset-member-index'
def find_named(root, name, cap=8000):
    hits = []
    n = 0
    for dirpath, dirnames, filenames in os.walk(root):
        n += len(filenames)
        if name in filenames:
            hits.append(Path(dirpath) / name)
        if len(hits) >= 5 or n >= cap:
            break
    print('  walked', n, 'files')
    return hits
print('=== ODA LAB pack walk ===')
receipt = ['# ODA LAB receipt', 'job=' + JOB, '']
for p in PACKS:
    name = p['name']
    print('---', p['n'], name, 'claimed', p['bytes'])
    hits = find_named(ROOT, name) if ROOT.exists() else []
    if not hits:
        print('  not on this MyDrive walk')
        receipt.append('- ' + name + ': NOT FOUND on mount')
        continue
    for hit in hits:
        sz = hit.stat().st_size
        print('  hit', hit, 'size', sz)
        receipt.append('- ' + name + ': ' + str(hit) + ' size=' + str(sz))
        if name.endswith('.zip') and sz < 20_000_000:
            r = subprocess.run(['unzip', '-l', str(hit)], capture_output=True, text=True)
            print((r.stdout or r.stderr)[:4000])
            receipt.append('```')
            receipt.append((r.stdout or r.stderr)[:2000])
            receipt.append('```')
print('=== done walk ===')
RECEIPT = '\n'.join(receipt)
print(RECEIPT)


In [ ]:
# Cell 3 — write receipt onto Drive if we can see the shelf name
out_dir = None
for cand in ROOT.rglob('12_ODA-LAB-NOTEBOOKS'):
    if cand.is_dir():
        out_dir = cand
        break
if out_dir is None:
    out_dir = ROOT / '12_ODA-LAB-NOTEBOOKS_LOCAL'
    out_dir.mkdir(exist_ok=True)
    print('shelf not found, wrote local fallback', out_dir)
else:
    print('shelf', out_dir)
from datetime import datetime
fn = datetime.now().strftime('%Y%m%d-%H%M') + '_ODA-LAB_sunset-member-index.RECEIPT.md'
path = out_dir / fn
path.write_text(RECEIPT)
print('WROTE', path)
